# Práctica 08 — Contenedores y CI/CD

**Curso:** MLOps y Monitorización con Python · **Clase 08** · **Tiempo estimado:** 40-55 minutos
**Notebook de clase:** `notebooks_clase/clase08_contenedores_cicd.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio parte de una pequeña función incompleta, inspirada directamente en el razonamiento del notebook de clase (validar instrucciones de un Dockerfile, comprobar el orden, detectar la trampa de `on:` en YAML, decidir si se promueve un modelo, contar pasos de un flujo). Aquí no se construye ninguna imagen Docker ni se ejecuta ningún flujo real: son funciones de Python pequeñas, para practicar el razonamiento sin depender de Docker, GitHub ni MLflow.
- Cada ejercicio tiene una o varias celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. No hace falta ninguna librería externa.

---
## Celda 0 — Setup

In [ ]:
print("Librerías cargadas correctamente (esta práctica no necesita instalar nada nuevo).")

## Ejercicio 1 — ¿Es una instrucción de Dockerfile válida?

Como `INSTRUCCIONES_VALIDAS` de la sección 4.1 del notebook de clase. Completa `es_instruccion_valida(palabra, validas)`, que recibe una palabra (por ejemplo `"FROM"`) y un conjunto de instrucciones válidas, y devuelve `True` si `palabra` está en `validas`, `False` en caso contrario.

In [ ]:
def es_instruccion_valida(palabra, validas):
    return False      # TODO: devuelve True si palabra está en validas, False si no

# Comprobación automática
VALIDAS_PRUEBA = {"FROM", "COPY", "RUN", "CMD"}
assert es_instruccion_valida("FROM", VALIDAS_PRUEBA) is True, "Completa las líneas marcadas con TODO"
assert es_instruccion_valida("MAINTAINER", VALIDAS_PRUEBA) is False, "Completa las líneas marcadas con TODO"
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — ¿El Dockerfile termina bien?

Como la comprobación de la sección 4.1 del notebook de clase (`instrucciones[-1] not in ("CMD", "ENTRYPOINT")`). Completa `termina_bien(instrucciones)`, que recibe una lista de instrucciones (cadenas, en orden) y devuelve `True` si la ÚLTIMA es `"CMD"` o `"ENTRYPOINT"`, `False` en caso contrario (también si la lista está vacía).

In [ ]:
def termina_bien(instrucciones):
    return True      # TODO: devuelve instrucciones and instrucciones[-1] in ("CMD", "ENTRYPOINT")

# Comprobación automática
assert termina_bien(["FROM", "COPY", "RUN", "CMD"]) is True, "Completa las líneas marcadas con TODO"
assert termina_bien(["FROM", "COPY", "RUN"]) is False, "Completa las líneas marcadas con TODO"
assert termina_bien([]) is False, "Completa las líneas marcadas con TODO"
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — La trampa de `on:` en YAML

Como la sección 5.1 del notebook de clase: en YAML 1.1, la clave `on:` sin comillas se interpreta como el booleano `True`. Completa `clave_disparadores_presente(flujo)`, que recibe el diccionario que devuelve `yaml.safe_load(...)` y devuelve `True` si `"on"` está entre las claves O si `True` está entre las claves (las dos formas posibles), `False` si ninguna de las dos está.

In [ ]:
def clave_disparadores_presente(flujo):
    return False      # TODO: devuelve "on" in flujo or True in flujo

# Comprobación automática
assert clave_disparadores_presente({True: {"push": None}, "name": "CI"}) is True, "Completa las líneas marcadas con TODO"
assert clave_disparadores_presente({"on": {"push": None}, "name": "CI"}) is True, "Completa las líneas marcadas con TODO"
assert clave_disparadores_presente({"name": "CI", "jobs": {}}) is False, "Completa las líneas marcadas con TODO"
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — ¿Se puede promover el candidato a producción?

Como la sección 7.2 del notebook de clase. Completa `se_puede_promover(tests_fallidos, f1_candidato, f1_produccion)`, que devuelve `True` solo si `tests_fallidos == 0` Y `f1_candidato >= f1_produccion`, `False` en caso contrario.

In [ ]:
def se_puede_promover(tests_fallidos, f1_candidato, f1_produccion):
    return True      # TODO: devuelve tests_fallidos == 0 and f1_candidato >= f1_produccion

# Comprobación automática
assert se_puede_promover(0, 0.32, 0.30) is True, "Completa las líneas marcadas con TODO"
assert se_puede_promover(1, 0.32, 0.30) is False, "Completa las líneas marcadas con TODO"
assert se_puede_promover(0, 0.20, 0.30) is False, "Completa las líneas marcadas con TODO"
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — Contar pasos que ejecutan un comando frente a pasos reutilizados

Como los `steps` de `ci.yml` (sección 5 del notebook de clase): algunos usan `run` (un comando de shell), otros usan `uses` (una acción reutilizada, como `actions/checkout@v4`). Completa `contar_pasos_run(pasos)`, que recibe una lista de diccionarios (cada uno con la clave `"run"` o la clave `"uses"`) y devuelve cuántos de ellos tienen la clave `"run"`.

In [ ]:
def contar_pasos_run(pasos):
    return 0      # TODO: devuelve sum(1 for p in pasos if "run" in p)

# Comprobación automática
PASOS_PRUEBA = [{"uses": "actions/checkout@v4"}, {"run": "pip install -r requirements.txt"}, {"run": "pytest tests/ -q"}]
assert contar_pasos_run(PASOS_PRUEBA) == 2, "Completa las líneas marcadas con TODO"
assert contar_pasos_run([{"uses": "actions/checkout@v4"}]) == 0, "Completa las líneas marcadas con TODO"
print("Ejercicio 5 correcto ✔")

---
## Reto final — Decidir el despliegue con un CD cauteloso

Integra el ejercicio 4 con la idea de la sección 8 (un CD cauteloso en ML añade una revisión humana). Completa `decidir_despliegue(tests_fallidos, f1_candidato, f1_produccion, revisado_por_humano)`, que devuelve `"rechazar"` si `se_puede_promover(...)` es `False`; si `se_puede_promover(...)` es `True` pero `revisado_por_humano` es `False`, devuelve `"esperar_revision"`; si `se_puede_promover(...)` es `True` Y `revisado_por_humano` es `True`, devuelve `"promover"`.

In [ ]:
def decidir_despliegue(tests_fallidos, f1_candidato, f1_produccion, revisado_por_humano):
    return "rechazar"      # TODO: usa se_puede_promover(...) y distingue los tres casos

# Comprobación automática
assert decidir_despliegue(0, 0.32, 0.30, True) == "promover", "Completa las líneas marcadas con TODO"
assert decidir_despliegue(0, 0.32, 0.30, False) == "esperar_revision", "Completa las líneas marcadas con TODO"
assert decidir_despliegue(1, 0.32, 0.30, True) == "rechazar", "Completa las líneas marcadas con TODO"
assert decidir_despliegue(0, 0.20, 0.30, True) == "rechazar", "Completa las líneas marcadas con TODO"
print("Reto final correcto ✔")

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
def es_instruccion_valida(palabra, validas):
    return palabra in validas

# Comprobación automática
VALIDAS_PRUEBA = {"FROM", "COPY", "RUN", "CMD"}
assert es_instruccion_valida("FROM", VALIDAS_PRUEBA) is True, "Completa las líneas marcadas con TODO"
assert es_instruccion_valida("MAINTAINER", VALIDAS_PRUEBA) is False, "Completa las líneas marcadas con TODO"
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
def termina_bien(instrucciones):
    return bool(instrucciones) and instrucciones[-1] in ("CMD", "ENTRYPOINT")

# Comprobación automática
assert termina_bien(["FROM", "COPY", "RUN", "CMD"]) is True, "Completa las líneas marcadas con TODO"
assert termina_bien(["FROM", "COPY", "RUN"]) is False, "Completa las líneas marcadas con TODO"
assert termina_bien([]) is False, "Completa las líneas marcadas con TODO"
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
def clave_disparadores_presente(flujo):
    return "on" in flujo or True in flujo

# Comprobación automática
assert clave_disparadores_presente({True: {"push": None}, "name": "CI"}) is True, "Completa las líneas marcadas con TODO"
assert clave_disparadores_presente({"on": {"push": None}, "name": "CI"}) is True, "Completa las líneas marcadas con TODO"
assert clave_disparadores_presente({"name": "CI", "jobs": {}}) is False, "Completa las líneas marcadas con TODO"
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
def se_puede_promover(tests_fallidos, f1_candidato, f1_produccion):
    return tests_fallidos == 0 and f1_candidato >= f1_produccion

# Comprobación automática
assert se_puede_promover(0, 0.32, 0.30) is True, "Completa las líneas marcadas con TODO"
assert se_puede_promover(1, 0.32, 0.30) is False, "Completa las líneas marcadas con TODO"
assert se_puede_promover(0, 0.20, 0.30) is False, "Completa las líneas marcadas con TODO"
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
def contar_pasos_run(pasos):
    return sum(1 for p in pasos if "run" in p)

# Comprobación automática
PASOS_PRUEBA = [{"uses": "actions/checkout@v4"}, {"run": "pip install -r requirements.txt"}, {"run": "pytest tests/ -q"}]
assert contar_pasos_run(PASOS_PRUEBA) == 2, "Completa las líneas marcadas con TODO"
assert contar_pasos_run([{"uses": "actions/checkout@v4"}]) == 0, "Completa las líneas marcadas con TODO"
print("Ejercicio 5 correcto ✔")

### Solución del reto final

In [ ]:
def decidir_despliegue(tests_fallidos, f1_candidato, f1_produccion, revisado_por_humano):
    if not se_puede_promover(tests_fallidos, f1_candidato, f1_produccion):
        return "rechazar"
    return "promover" if revisado_por_humano else "esperar_revision"

# Comprobación automática
assert decidir_despliegue(0, 0.32, 0.30, True) == "promover", "Completa las líneas marcadas con TODO"
assert decidir_despliegue(0, 0.32, 0.30, False) == "esperar_revision", "Completa las líneas marcadas con TODO"
assert decidir_despliegue(1, 0.32, 0.30, True) == "rechazar", "Completa las líneas marcadas con TODO"
assert decidir_despliegue(0, 0.20, 0.30, True) == "rechazar", "Completa las líneas marcadas con TODO"
print("Reto final correcto ✔")

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto de la clase no te ha quedado claro (Dockerfile, `ci.yml`, registro de modelos de MLflow, CI frente a CD)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*